In [1]:
!pip install pandas seaborn scikit-learn

Defaulting to user installation because normal site-packages is not writeable


In [2]:
import pandas as pd
import numpy as np
import seaborn as sns
from sklearn.linear_model import LogisticRegression

In [3]:
df = sns.load_dataset("titanic")
print(df.head())

   survived  pclass     sex   age  sibsp  parch     fare embarked  class  \
0         0       3    male  22.0      1      0   7.2500        S  Third   
1         1       1  female  38.0      1      0  71.2833        C  First   
2         1       3  female  26.0      0      0   7.9250        S  Third   
3         1       1  female  35.0      1      0  53.1000        S  First   
4         0       3    male  35.0      0      0   8.0500        S  Third   

     who  adult_male deck  embark_town alive  alone  
0    man        True  NaN  Southampton    no  False  
1  woman       False    C    Cherbourg   yes  False  
2  woman       False  NaN  Southampton   yes   True  
3  woman       False    C  Southampton   yes  False  
4    man        True  NaN  Southampton    no   True  


EDA

In [4]:
print(df.columns)

Index(['survived', 'pclass', 'sex', 'age', 'sibsp', 'parch', 'fare',
       'embarked', 'class', 'who', 'adult_male', 'deck', 'embark_town',
       'alive', 'alone'],
      dtype='object')


In [8]:
df.shape

(891, 15)

In [9]:
df.info

<bound method DataFrame.info of      survived  pclass     sex   age  sibsp  parch     fare embarked   class  \
0           0       3    male  22.0      1      0   7.2500        S   Third   
1           1       1  female  38.0      1      0  71.2833        C   First   
2           1       3  female  26.0      0      0   7.9250        S   Third   
3           1       1  female  35.0      1      0  53.1000        S   First   
4           0       3    male  35.0      0      0   8.0500        S   Third   
..        ...     ...     ...   ...    ...    ...      ...      ...     ...   
886         0       2    male  27.0      0      0  13.0000        S  Second   
887         1       1  female  19.0      0      0  30.0000        S   First   
888         0       3  female   NaN      1      2  23.4500        S   Third   
889         1       1    male  26.0      0      0  30.0000        C   First   
890         0       3    male  32.0      0      0   7.7500        Q   Third   

       who  adult_m

In [10]:
df.describe()

,survived,pclass,age,sibsp,parch,fare
count,891.000000,891.000000,714.000000,891.000000,891.000000,891.000000
mean,0.383838,2.308642,29.699118,0.523008,0.381594,32.204208
std,0.486592,0.836071,14.526497,1.102743,0.806057,49.693429
min,0.000000,1.000000,0.420000,0.000000,0.000000,0.000000
25%,0.000000,2.000000,20.125000,0.000000,0.000000,7.910400
50%,0.000000,3.000000,28.000000,0.000000,0.000000,14.454200
75%,1.000000,3.000000,38.000000,1.000000,0.000000,31.000000
max,1.000000,3.000000,80.000000,8.000000,6.000000,512.329200


In [11]:
df.isnull().sum()

survived         0
pclass           0
sex              0
age            177
sibsp            0
parch            0
fare             0
embarked         2
class            0
who              0
adult_male       0
deck           688
embark_town      2
alive            0
alone            0
dtype: int64

In [12]:
df["survived"]

0      0
1      1
2      1
3      1
4      0
      ..
886    0
887    1
888    0
889    1
890    0
Name: survived, Length: 891, dtype: int64

In [13]:
df['survived'].value_counts()

survived
0    549
1    342
Name: count, dtype: int64

In [14]:
df['survived'].value_counts(normalize=True)

survived
0    0.616162
1    0.383838
Name: proportion, dtype: float64

In [15]:
df.groupby('sex')['survived'].mean()

sex
female    0.742038
male      0.188908
Name: survived, dtype: float64

Supervivencia por SEXO y CLASE

In [16]:
df.groupby('sex')['survived'].mean()

sex
female    0.742038
male      0.188908
Name: survived, dtype: float64

In [17]:
df.groupby('pclass')['survived'].mean()

pclass
1    0.629630
2    0.472826
3    0.242363
Name: survived, dtype: float64

Limpieza y Preparacion

In [18]:
features = ['pclass', 'sex', 'age', 'fare', 'embarked']
target = 'survived'

In [19]:
df_model = df[features + [target]].copy()

In [20]:
print(df_model.head())

   pclass     sex   age     fare embarked  survived
0       3    male  22.0   7.2500        S         0
1       1  female  38.0  71.2833        C         1
2       3  female  26.0   7.9250        S         1
3       1  female  35.0  53.1000        S         1
4       3    male  35.0   8.0500        S         0


In [23]:
df_model.isnull().sum()

pclass        0
sex           0
age         177
fare          0
embarked      2
survived      0
dtype: int64

Rellenar valores Null con media y moda

In [24]:
df_model['age'] = df_model['age'].fillna(df_model['age'].median())
df_model['embarked'] = df_model['embarked'].fillna(df_model['embarked'].mode()[0])

In [25]:
df_model.isnull().sum()

pclass      0
sex         0
age         0
fare        0
embarked    0
survived    0
dtype: int64

Convertir Texto A Numeros

In [26]:
df_model = pd.get_dummies(
    df_model, 
    columns=['embarked', 'sex'],
    drop_first=True 
)

In [27]:
df_model.head()

,pclass,age,fare,survived,embarked_Q,embarked_S,sex_male
0,3,22.0,7.2500,0,False,True,True
1,1,38.0,71.2833,1,False,False,False
2,3,26.0,7.9250,1,False,True,False
3,1,35.0,53.1000,1,False,True,False
4,3,35.0,8.0500,0,False,True,True


Separacion X - Y

In [28]:
#features
x = df_model.drop(['survived'], axis= 1)
#target
y = df_model['survived']

TRAIN/TEST SPLIT

In [29]:
from sklearn.model_selection import train_test_split
x_train, x_test, y_train, y_test = train_test_split(
    x, 
    y, 
    test_size= 0.2,
    random_state=42
)

In [31]:
print(x_train.shape)
print(x_test.shape)
print(y_train.shape)
print(y_test.shape)

(712, 6)
(179, 6)
(712,)
(179,)


Escalamientot de los Datos

In [37]:
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
x_train_scaled = scaler.fit_transform(x_train)
x_test_scaled = scaler.transform(x_test)

In [39]:
print(x_train_scaled[:5])

[[-1.61413602  1.25364106 -0.07868358 -0.30335547  0.59248936  0.7243102 ]
 [-0.40055118 -0.47728355 -0.37714494 -0.30335547  0.59248936  0.7243102 ]
 [ 0.81303367  0.21508629 -0.47486697 -0.30335547  0.59248936  0.7243102 ]
 [ 0.81303367 -0.24649361 -0.47623026 -0.30335547  0.59248936  0.7243102 ]
 [ 0.81303367 -1.78509326 -0.02524937 -0.30335547  0.59248936 -1.38062393]]


TRAIN

In [41]:
model = LogisticRegression()
model.fit(x_train_scaled, y_train)

,penalty,'l2'
,dual,False
,tol,0.0001
,C,1.0
,fit_intercept,True
,intercept_scaling,1
,class_weight,None
,random_state,None
,solver,'lbfgs'
,max_iter,100
,multi_class,'deprecated'


In [42]:
# Predicciones
y_pred = model.predict(x_test_scaled)

In [43]:
print(y_pred[:10])

[0 0 0 1 1 1 1 0 1 1]


In [44]:
print(y_test[:10].values)

[1 0 0 1 1 1 1 0 1 1]


9 / 10

In [47]:
# Accuaracy 
from sklearn.metrics import accuracy_score
accuracy = accuracy_score(y_test, y_pred)
print("Accuracy: ", accuracy)

Accuracy:  0.7988826815642458


METRICAS

In [48]:
from sklearn.metrics import confusion_matrix, classification_report

cm = confusion_matrix(y_test, y_pred)
print(cm)

print(classification_report(y_test, y_pred))

[[88 17]
 [19 55]]
              precision    recall  f1-score   support

           0       0.82      0.84      0.83       105
           1       0.76      0.74      0.75        74

    accuracy                           0.80       179
   macro avg       0.79      0.79      0.79       179
weighted avg       0.80      0.80      0.80       179



Prediccion de un dato Inventado

In [51]:
print(x.columns)

Index(['pclass', 'age', 'fare', 'embarked_Q', 'embarked_S', 'sex_male'], dtype='object')


In [55]:
#datos scalados
nuevo_pasajero = pd.DataFrame({
    'pclass': [1],
    'age': [23],
    'fare': [120],
    'embarked_Q': [0],
    'embarked_S': [1],
    'sex_male': [1]
})
nuevo_pasajero_scaled = scaler.transform(nuevo_pasajero)
#prediccion de sus datos
prediccion = model.predict(nuevo_pasajero_scaled)
probabilidad = model.predict_proba(nuevo_pasajero_scaled)
print(prediccion)
print(probabilidad)

[0]
[[0.53959962 0.46040038]]
